In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

# === Load your data ===
df = pd.read_csv("Figure_2B_GIN440_HAP1_TSC22D1,2_PIScores.csv")

# === Prepare columns ===
if 'Gene rank' in df.columns:
    gene_rank = df['Gene rank']
elif 'Rank' in df.columns:
    gene_rank = df['Rank']
else:
    gene_rank = np.arange(1, len(df) + 1)

x_vals = gene_rank / 1e3
piscore = df['PIScore']
fdr = df['FDR']
size = -np.log10(fdr)

# === Separate classes ===
base_color = 'white'

x_base = x_vals
y_base = piscore
s_base = size * 150

# Significant (negative and positive)
mask_neg = (piscore < -0.4) & (fdr < 0.25)
mask_pos = (piscore > 0.4) & (fdr < 0.25)

x_neg = x_vals[mask_neg]
y_neg = piscore[mask_neg]
s_neg = size[mask_neg] * 150

x_pos = x_vals[mask_pos]
y_pos = piscore[mask_pos]
s_pos = size[mask_pos] * 150

# TSC22D genes
mask_tsc = df['Gene'].astype(str).str.contains('TSC22D')
x_tsc = x_vals[mask_tsc]
y_tsc = piscore[mask_tsc]
s_tsc = size[mask_tsc] * 150

# === Plot ===
plt.figure(figsize=(3, 7))

# 1. White background for all
plt.scatter(
    x_base, y_base,
    s=s_base,
    c=base_color,
    edgecolor='#b3b3b3',
    linewidth=0.1,
    marker='o',
    zorder=1
)

# 2. Significant dots on top
plt.scatter(
    x_neg, y_neg,
    s=s_neg,
    c='#49c1bb',
    edgecolor='black',
    linewidth=0.1,
    marker='o',
    zorder=2
)

plt.scatter(
    x_pos, y_pos,
    s=s_pos,
    c='#bababa',
    edgecolor='black',
    linewidth=0.1,
    marker='o',
    zorder=2
)

# 3. TSC22D genes last (on top of everything)
plt.scatter(
    x_tsc, y_tsc,
    s=s_tsc,
    c='#ffe5b6',
    edgecolor='black',
    linewidth=0.1,
    marker='o',
    zorder=3
)

# Horizontal line
plt.axhline(0, color='#bababa', linestyle='--', linewidth=1, zorder=0)

# Fonts
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.xlabel('Gene rank (x10³)', fontname='Arial')
plt.ylabel('PIScore', fontname='Arial')
plt.title('Gene rank vs PIScore', fontname='Arial')
plt.xticks(fontname='Arial')
plt.yticks(fontname='Arial')

# Save
plt.savefig("Figure_2B_Scatter_plot_TSC22D1,2-KO_CRISPR_screen.pdf", format="pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.tight_layout()
plt.show()


In [ ]:
# legend_panel.py
# Standalone legend figure for:
# - Color coding of dots
# - Dot sizes mapped to -log10(FDR)
#
# Output: legend_panel.png (transparent), legend_panel.svg

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import rcParams

# ========== CONFIG ==========
# Font (Arial if available, fallback to DejaVu Sans)
rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
rcParams['font.size'] = 11

# Colors per your spec
COLOR_NEG = "#49c1bb"   # PI < -0.4 & FDR < 0.25
COLOR_POS = "#bababa"   # PI > +0.4 & FDR < 0.25
COLOR_TSC = "#ffe5b6"   # TSC22D genes override
COLOR_BASE = "white"    # background dots (not shown in legend)

# Dot outline
EDGE_COLOR = "black"
EDGE_WIDTH = 0.8

# Size scale: same as your main plot (s = -log10(FDR) * SIZE_SCALE)
SIZE_SCALE = 50.0

# Pick a few representative FDRs for the size legend
FDR_VALUES = [0.1, 0.01, 0.001]  # adjust as you like
# ============================

def make_marker_handle(color, label=None, size_pts=80):
    """Return a circular legend handle with black edge."""
    return Line2D(
        [], [], 
        marker='o', linestyle='',
        markersize=np.sqrt(size_pts),  # Line2D uses points^2 relation differently than scatter; sqrt keeps it intuitive
        markerfacecolor=color, markeredgecolor=EDGE_COLOR, markeredgewidth=EDGE_WIDTH,
        label=label
    )

def main():
    # Create a compact figure; we'll place two grouped legends
    fig = plt.figure(figsize=(4.6, 2.8))  # tune width/height to your layout
    ax = fig.add_axes([0, 0, 1, 1])       # full-figure axis
    ax.axis('off')

    # --- Color legend (three categories) ---
    color_handles = [
        make_marker_handle(COLOR_NEG,  'PI < -0.4, FDR < 0.25', size_pts=110),
        make_marker_handle(COLOR_POS,  'PI > +0.4, FDR < 0.25', size_pts=110),
        make_marker_handle(COLOR_TSC,  'TSC22D genes',          size_pts=110),
    ]
    color_legend = ax.legend(
        handles=color_handles,
        title="Color coding",
        loc="center left",
        bbox_to_anchor=(0.02, 0.55),
        frameon=True
    )
    ax.add_artist(color_legend)

    # --- Size legend (FDR mapping) ---
    size_handles = []
    size_labels  = []
    for fdr in FDR_VALUES:
        s = -np.log10(fdr) * SIZE_SCALE
        size_handles.append(make_marker_handle("white", size_pts=s))
        size_labels.append(f"FDR = {fdr:g}")

    size_legend = ax.legend(
        handles=size_handles,
        labels=size_labels,
        title="-log₁₀(FDR) dot size",
        loc="center left",
        bbox_to_anchor=(0.02, 0.08),
        frameon=True
    )

    # Tight-ish layout without cropping legends
    plt.savefig("legend_panel.png", dpi=400, transparent=True, bbox_inches="tight", pad_inches=0.15)
    plt.savefig("legend_panel.svg", transparent=True, bbox_inches="tight", pad_inches=0.15)
    print("Saved: legend_panel.png and legend_panel.svg")

if __name__ == "__main__":
    main()
